# Connectome Hypothesis Testing: EM2 False Synapses (Dedicated Run)

**Evaluates False Synapses (EM2: False Positives / Added Edges)** on randomized degree/weight-preserved null graphs in `null_only` mode.

Pre-computes candidate edges via `CandidateGenerator` before hypothesis execution.


In [ ]:
# Cell 1: Environment Setup & sys.path Discovery
# ============================================================
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

# Locate codebase root automatically
CODEBASE_ROOT = None
possible_code_paths = [
    Path('/kaggle/input/datasets/jeet7771/flywire-hypothesis-kaggle-package'),
    Path('/kaggle/input/datasets/jeet7771/flywire-codebase'),
    Path('/kaggle/input/flywire-codebase'),
    Path('/kaggle/working'),
    Path(os.getcwd()).resolve(),
]

for p in possible_code_paths:
    if (p / 'hypothesis_testing').exists() and (p / 'configs').exists():
        CODEBASE_ROOT = p
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print(f'[OK] Discovered codebase at: {CODEBASE_ROOT}')
        break

if CODEBASE_ROOT is None:
    # Fallback to current working directory
    CODEBASE_ROOT = Path(os.getcwd()).resolve()
    sys.path.insert(0, str(CODEBASE_ROOT))
    print(f'[WARN] Using fallback codebase path: {CODEBASE_ROOT}')

CONFIGS_ROOT = str(CODEBASE_ROOT / 'configs')
print(f'[OK] Configs root : {CONFIGS_ROOT}')

# Locate raw datasets folder
KAGGLE_DATA_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')
if KAGGLE_DATA_PATH.exists():
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
    print(f'[OK] Dataset root : {DATASET_ROOT}')
else:
    DATASET_ROOT = 'research_data/raw'
    print(f'[INFO] Dataset root (local/fallback): {DATASET_ROOT}')

print(f'Environment: {"KAGGLE" if IS_KAGGLE else "LOCAL"}')


In [ ]:
# Cell 2: Framework Imports & Real-time Logging Setup
# ============================================================
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
    force=True
)

import warnings
warnings.filterwarnings('ignore')

import time
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from core.data_loader import load_dataset
from core.graph_builder import GraphBuilder
from modules.preprocessing import CandidateGenerator, preprocess_graph
from hypothesis_testing.config import HypothesisExperimentConfig, ExecutionMode
from hypothesis_testing.runners.hypothesis_experiment_runner import HypothesisExperimentRunner

print('[OK] Hypothesis-testing framework & real-time logging initialized successfully.')


In [ ]:
# ============================================================
# Cell 3: RUNTIME CONFIGURATION (EM2 False Synapses)
# ============================================================

EXECUTION_MODE = 'null_only'
DATASET_NAME = 'BANC'
NULL_MODEL_NAME = 'degree_preserving'

# Single null graph topology
NULL_GRAPH_SEEDS = [1]

# Error model 2 only
ERROR_MODELS = [
    'false_synapses',
]

# Error rates (10 rates)
ERROR_RATES = [0.000, 0.005, 0.010, 0.020, 0.030, 0.050, 0.075, 0.100, 0.150, 0.200]

# Perturbation trial seeds (5 replicates on the null graph)
RANDOM_SEEDS = [1, 2, 3, 4, 5]

ANALYSES = [
    'basic_structure',
    'degree_distribution',
    'connected_components',
    'reciprocity',
    'pagerank',
]

OUTPUT_ROOT = Path('results') / 'hypothesis_testing'

print(f'Execution Mode    : {EXECUTION_MODE}')
print(f'Dataset Name      : {DATASET_NAME}')
print(f'Error Models      : {ERROR_MODELS}')
print(f'Null Graph Seeds  : {NULL_GRAPH_SEEDS}')
print(f'Perturbation Seeds: {RANDOM_SEEDS}')


In [ ]:
# ============================================================
# Cell 4: Candidate Generation for False Synapses
# ============================================================
cache_dir = Path('research_data/cache/false_synapses')
cache_dir.mkdir(parents=True, exist_ok=True)
cache_path = cache_dir / 'candidates.parquet'

if cache_path.exists():
    cand_table = pl.read_parquet(str(cache_path))
    print(f'[OK] Existing candidate table found with {len(cand_table):,} candidate pairs at {cache_path}.')
else:
    print(f'[INFO] Building candidate table for dataset "{DATASET_NAME}"...')
    t0 = time.perf_counter()
    raw_dataset = load_dataset(dataset_name=DATASET_NAME, dataset_root=DATASET_ROOT, configs_root=CONFIGS_ROOT)
    raw_graph = GraphBuilder().build(raw_dataset)
    prepared = preprocess_graph(raw_graph, index_node_attrs=['top_region'])
    generator = CandidateGenerator(prepared)
    generator.generate(cache_path)
    cand_table = pl.read_parquet(str(cache_path))
    print(f'[OK] Generated {len(cand_table):,} candidates in {time.perf_counter() - t0:.1f} s → {cache_path}')


In [ ]:
# Cell 5: Assemble Config & Validate
# ============================================================
exp_config = HypothesisExperimentConfig(
    dataset_name=DATASET_NAME,
    dataset_root=DATASET_ROOT,
    configs_root=CONFIGS_ROOT,
    execution_mode=EXECUTION_MODE,
    null_model_name=NULL_MODEL_NAME,
    null_graph_seeds=NULL_GRAPH_SEEDS,
    error_model_names=ERROR_MODELS,
    error_model_configs={'false_synapses': {'candidate_cache_path': str(cache_path)}},
    error_rates=ERROR_RATES,
    random_seeds=RANDOM_SEEDS,
    analysis_names=ANALYSES,
    output_root=str(OUTPUT_ROOT),
)

print(f'[OK] Experiment configuration prepared for mode: {exp_config.execution_mode.value}.')


In [ ]:
# Cell 6: Execute Hypothesis Testing Pipeline
# ============================================================
runner = HypothesisExperimentRunner()
result = runner.run(exp_config)

print(f'Execution Status: {result.status}')
print(f'Total Runtime   : {result.runtime_seconds:.2f} seconds')
print(f'Deliverables    : {list(result.exported_paths.keys())}')


In [ ]:
# Cell 7: Inspect Generated Replicate-Level Outputs
# ============================================================
null_csv = OUTPUT_ROOT / DATASET_NAME / 'null_observations' / 'replicate_level_effects.csv'
if null_csv.exists():
    df_null = pd.read_csv(null_csv)
    print(f'[OK] Replicate records exported: {len(df_null)} rows')
    display(df_null.head(10))
else:
    print('Replicate CSV not found.')


In [ ]:
# Cell 8: Package Deliverables for 1-Click Download
# ============================================================
import shutil
out_folder = OUTPUT_ROOT / DATASET_NAME
zip_name = f'hypothesis_testing_{DATASET_NAME}_em2_null_results'
if out_folder.exists():
    shutil.make_archive(zip_name, 'zip', str(out_folder))
    print(f'[OK] Created downloadable archive: {zip_name}.zip')
else:
    print('Output directory not found for archiving.')
